# Experimento pareado: dropout en la RNN32 con selección F2

Se comparan dos entrenamientos nuevos con los mismos datos, particiones, RNN32, semilla, optimizer, BCE, early stopping y selección de umbral. La única diferencia es `dropout_rate=0.0` frente a `0.2`, aplicado después de ReLU en la capa de fusión.

El entrenamiento requiere GPU CUDA y se detiene si no está disponible. Train usa 2022–2023, validación 2024 y 2025 sigue reservada.

## 1. Subir y verificar el paquete

En la raíz del repositorio genera `runs/rnn_colab_dropout_v3.zip` con `python scripts/empaquetar_experimento_dropout.py`. Guarda el SHA-256 impreso y sube el ZIP al panel **Archivos** de Colab. Selecciona una GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.

In [ ]:
from pathlib import Path
import hashlib, json, stat, subprocess, sys, zipfile
from google.colab import files

ROOT = Path('/content/f1-dropout')
PACKAGE = Path('/content/rnn_colab_dropout_v3.zip')
EXPECTED_PACKAGE_SHA256 = ''  # Pega aquí el hash mostrado por el empaquetador local.
if ROOT.exists(): raise FileExistsError('Usa una sesión limpia: /content/f1-dropout ya existe.')
if not PACKAGE.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Sube exactamente el ZIP del experimento dropout.')
    name, content = next(iter(uploaded.items()))
    if Path(name).name != 'rnn_colab_dropout_v3.zip': raise ValueError('Nombre ZIP inesperado.')
    PACKAGE.write_bytes(content)
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
package_hash = sha256(PACKAGE)
print('SHA-256 del paquete:', package_hash)
if not EXPECTED_PACKAGE_SHA256.strip(): raise ValueError('Pega primero el SHA-256 mostrado por el empaquetador local.')
if package_hash != EXPECTED_PACKAGE_SHA256.strip().lower(): raise ValueError('El hash no coincide con el ZIP local.')
with zipfile.ZipFile(PACKAGE) as z:
    members=z.infolist(); names=set()
    for m in members:
        p=Path(m.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in m.filename or stat.S_ISLNK(m.external_attr >> 16): raise ValueError(f'Ruta ZIP no segura: {m.filename}')
        if m.filename in names: raise ValueError(f'Miembro duplicado: {m.filename}')
        names.add(m.filename)
    manifest=json.loads(z.read('colab_package_manifest.json'))
    if manifest.get('artifact_version') != 'rnn_colab_dropout_inputs_v3' or manifest.get('test_arrays_included') is not False: raise ValueError('Contrato de paquete no admitido o contiene test.')
    expected=set(manifest['files_sha256']) | {'colab_package_manifest.json'}
    if names != expected: raise ValueError('Los archivos del ZIP no coinciden con su manifiesto.')
    ROOT.mkdir(parents=True)
    z.extractall(ROOT)
for name, digest in manifest['files_sha256'].items():
    if sha256(ROOT/name) != digest: raise ValueError(f'SHA-256 incorrecto: {name}')
print('Manifiesto verificado:', len(manifest['files_sha256']), 'archivos; sin datos de prueba.')

## 2. Instalar dependencias y confirmar GPU

La comparación se cancela antes de entrenar si Colab no expone CUDA. No cambies a CPU ni cambies el tipo de entorno entre los dos modelos.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-colab-experiments.txt')], check=True)
import torch
cuda_available = torch.cuda.is_available()
print('PyTorch:', torch.__version__, '| build CUDA:', torch.version.cuda, '| cuda.is_available:', cuda_available, '| dispositivos visibles:', torch.cuda.device_count())
if not cuda_available:
    if torch.version.cuda is None:
        detail = 'Este runtime tiene un build CPU de PyTorch.'
    else:
        detail = 'PyTorch tiene soporte CUDA, pero no ve un dispositivo GPU conectado.'
    raise RuntimeError(detail + ' Elimina el runtime actual, reconecta una sesión Colab GPU y vuelve a ejecutar; el notebook no instalará ni sustituirá torch.')
print('GPU:', torch.cuda.get_device_name(0))

## 3. Entrenar la pareja

Cada hijo aplica early stopping sobre F2 de validación a umbral 0,5, con paciencia seis y desempate por BCE menor. Con el checkpoint congelado, selecciona el umbral F2 en la cuadrícula ya acordada. El reporte conserva F2/F1, precision, recall, AP, BCE, matrices de confusión y alertas.

In [ ]:
RUN = ROOT/'runs/rnn32_dropout02_f2_v2_seed42'
if RUN.exists() or RUN.with_suffix('.zip').exists(): raise FileExistsError('La salida ya existe. Conserva los artefactos y usa una sesión/salida nueva.')
subprocess.run([sys.executable, 'scripts/entrenar_comparacion_dropout.py', '--reference-config', 'configs/experimento_dropout_ref_v1.json', '--candidate-config', 'configs/experimento_dropout_02_v1.json', '--output', str(RUN.relative_to(ROOT))], cwd=ROOT, check=True)
summary=json.loads((RUN/'pair_summary.json').read_text())
print(json.dumps(summary, indent=2))
files.download(str(RUN.with_suffix('.zip')))

## 4. Devolver resultados al proyecto

Descarga `rnn32_dropout02_f2_v2_seed42.zip` y colócalo en la carpeta local `runs/` del repositorio. No extraigas ni alteres los archivos del paquete. La evaluación local CPU verificará hashes y paridad de probabilidades antes de comparar ambos modelos.